# CAASPP District Chart Maker

This notebook turns CAASPP files into chart-ready spreadsheets for Google Sheets.

You will do four things:

1. Upload the CAASPP ZIP files you downloaded.
2. Enter your district name and district codes.
3. Run the analysis.
4. Download the finished spreadsheet files.

**Privacy note:** files uploaded here go to your own temporary Colab session. They do not go to the repo owner’s Google Drive.


## Step 1 — Start the notebook

In Google Colab, click the play button next to the cell below.

You do not need to understand or edit the code. This just prepares the notebook.

**Screenshot suggestion:** add a screenshot showing the Colab play button.


In [ ]:
#@title Step 1: Prepare the notebook { display-mode: "form" }
try:
    import pandas as pd
    import openpyxl
except Exception:
    !pip -q install pandas openpyxl

from pathlib import Path
import sys

RAW_DIR = Path('data/raw')
WORK_DIR = Path('data/working')
OUTPUT_DIR = Path('outputs')
for folder in [RAW_DIR, WORK_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if Path('src').exists() and str(Path('src').resolve()) not in sys.path:
    sys.path.append(str(Path('src').resolve()))

print('Ready for Step 2.')


## Step 2 — Upload your CAASPP files

Download the **Smarter Balanced districtwide All Student Groups** files from CAASPP.

ZIP files are fine. You can upload the ZIPs directly.

Click the play button below, then choose all the CAASPP files for your district.

**Screenshot suggestion:** add a screenshot showing the file upload button.


In [ ]:
#@title Step 2: Upload CAASPP ZIP/TXT/CSV files { display-mode: "form" }
try:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        print('No files uploaded yet. Run this cell again when you are ready.')
    for name, data in uploaded.items():
        target = RAW_DIR / name
        target.write_bytes(data)
        print(f'Uploaded: {target}')
except ModuleNotFoundError:
    print('This upload button only appears in Google Colab.')
    print('If you are running locally, put CAASPP ZIP/TXT/CSV files in data/raw/.')

print()
print('Files currently ready:')
for p in sorted(RAW_DIR.glob('*')):
    print(' -', p.name)


## Step 3 — Enter your district information

Change the three boxes below for your district.

Examples:

- Las Virgenes Unified: county code `19`, district code `64683`
- San Marcos Unified: county code `37`, district code `73791`

Usually, leave the grades and student groups alone.

**Screenshot suggestion:** add a screenshot showing the three boxes filled in.


In [ ]:
#@title Step 3: District settings { display-mode: "form" }
DISTRICT_NAME = "Las Virgenes Unified" #@param {type:"string"}
COUNTY_CODE = "19" #@param {type:"string"}
DISTRICT_CODE = "64683" #@param {type:"string"}

# These defaults are designed for the standard SBS-style SED/NSED analysis.
GRADES = ['03', '04', '05', '06', '08', '11']
STUDENT_GROUPS = {'031': 'SED', '111': 'NSED'}
TESTS = {'01': 'ELA', '02': 'MATH'}
YEAR_ORDER = ['24-25', '23-24', '22-23', '21-22', '18-19', '17-18', '16-17', '15-16', '14-15']

DISTRICT_SLUG = DISTRICT_NAME.lower().replace(' ', '_').replace('/', '_')

print('District name:', DISTRICT_NAME)
print('County code:', COUNTY_CODE)
print('District code:', DISTRICT_CODE)
print('Ready for Step 4.')


## Step 4 — Run the analysis

Click the play button below.

This step reads the CAASPP files, pulls the districtwide SED and NSED rows, and makes chart-ready tables.

When it finishes, look for the sanity check. For the default setup, each year should usually have:

`6 grades × 2 groups × 2 subjects = 24 rows`

If a warning appears, use the audit file later to check what happened.


In [ ]:
#@title Step 4: Run the analysis { display-mode: "form" }
from caaspp_sbs_tools import (
    collect_input_files,
    process_caaspp_files,
    print_sanity_checks,
)

input_files = collect_input_files(RAW_DIR, WORK_DIR)
print('Files found:')
for p in input_files:
    print(' -', p.name)

wide, audit_long, gap_long, avg_gap_by_grade, band_gap_by_year = process_caaspp_files(
    input_files=input_files,
    county_code=COUNTY_CODE,
    district_code=DISTRICT_CODE,
    district_name=DISTRICT_NAME,
    grades=GRADES,
    student_groups=STUDENT_GROUPS,
    tests=TESTS,
    year_order=YEAR_ORDER,
)

print()
print('Sanity check:')
print_sanity_checks(audit_long, grades=GRADES)

print()
print('Done. Ready for Step 5.')


## Step 5 — Check the results

The tables below are previews.

The most important preview is **Average gap by grade**. It shows the average difference between NSED and SED students.

Gap means:

`NSED percent met/exceeded − SED percent met/exceeded`

So a gap of `20` means NSED students were 20 percentage points higher than SED students.


In [ ]:
#@title Step 5: Preview the chart tables { display-mode: "form" }
print('Wide time-series table:')
display(wide)

print('Average SED/NSED gap by grade:')
display(avg_gap_by_grade)

print('Lower grades vs upper grades:')
display(band_gap_by_year)

print('First rows of the audit table:')
display(audit_long.head(10))


## Step 6 — Create the files to download

Click the play button below.

This creates CSV files and one Excel workbook you can upload to Google Sheets.


In [ ]:
#@title Step 6: Create downloadable files { display-mode: "form" }
from caaspp_sbs_tools import export_outputs

paths = export_outputs(
    output_dir=OUTPUT_DIR,
    district_slug=DISTRICT_SLUG,
    wide=wide,
    audit_long=audit_long,
    gap_long=gap_long,
    avg_gap_by_grade=avg_gap_by_grade,
    band_gap_by_year=band_gap_by_year,
)

print('Created these files:')
for label, path in paths.items():
    print(f' - {path}')


## Step 7 — Download the finished files

Click the play button below.

Your browser should download the output files.

The easiest file to use is the Excel workbook ending in:

`chart_ready_outputs.xlsx`

Upload that workbook to Google Sheets.

**Screenshot suggestion:** add a screenshot showing the downloaded files.


In [ ]:
#@title Step 7: Download the files { display-mode: "form" }
try:
    from google.colab import files
    for path in paths.values():
        files.download(str(path))
except ModuleNotFoundError:
    print('Not running in Colab. Your files are in the outputs/ folder.')


## Step 8 — Make charts in Google Sheets

Upload the Excel workbook to Google Sheets.

### Chart idea 1: Proficiency over time

Use the sheet named **Wide Time Series**.

- X-axis: `School Year`
- Y-axis: `Percent Met or Exceeded Standard`

### Chart idea 2: SED/NSED gap by grade

Use the sheet named **Avg Gap by Grade**.

- X-axis: `Grade`
- Y-axis: `NSED–SED Gap, percentage points`

### Chart idea 3: Lower grades vs upper grades

Use the sheet named **Band Gap by Year**.

- X-axis: `School Year`
- Y-axis: `NSED–SED Gap, percentage points`

### Suggested caveat for presentations

These are grade-level snapshots, not the same students followed over time. Small districts and small SED subgroups can be noisy, so focus on patterns that repeat across grades, subjects, or years.
